# Vietnamese Depression Project — LDA Topic Modeling v2

**Notebook version:** 2.0  
**Primary implementation:** `lda_topic_modeling_v2.py`

This notebook is a lightweight interface around the frozen v2 script. The scientific protocol is in `LDA_V2_PROTOCOL.md`. The recommended server workflow is:

1. run **prepare-only**;
2. inspect family/corpus/dictionary audits;
3. only then launch the 60-run full search.

LDA topics are latent themes, not ground-truth classes. Do not force K=12 to match the GPT taxonomy.


## 1. Environment and project checks

In [ ]:
from pathlib import Path
import sys, subprocess, json
import pandas as pd

PROJECT_ROOT = Path.cwd()
SCRIPT = PROJECT_ROOT / 'lda_topic_modeling_v2.py'
print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)
print('Script exists:', SCRIPT.exists())
for required in [
    PROJECT_ROOT/'datasets'/'dataset_1.csv',
    PROJECT_ROOT/'datasets'/'dataset_2.csv',
    PROJECT_ROOT/'reproduction_results_v1'/'preprocessed'/'Dataset_depression_vi1.csv',
    PROJECT_ROOT/'reproduction_results_v1'/'preprocessed'/'Dataset_depression_vi2.csv',
]:
    print(required, 'OK' if required.exists() else 'MISSING')
if not SCRIPT.exists():
    raise FileNotFoundError(SCRIPT)


## 2. Dependency check

In [ ]:
import numpy, pandas, scipy, sklearn, gensim, matplotlib
print('numpy:', numpy.__version__)
print('pandas:', pandas.__version__)
print('scipy:', scipy.__version__)
print('sklearn:', sklearn.__version__)
print('gensim:', gensim.__version__)
print('matplotlib:', matplotlib.__version__)
print('DEPENDENCIES: PASS')


## 3. Prepare-only audit gate

Run this before the full search. It reconstructs and audits the canonical corpus, conservative near-duplicate families, topic lexical cleanup, and dictionary without training the 60 LDA models.

In [ ]:
RUN_PREPARE_ONLY = True

if RUN_PREPARE_ONLY:
    cmd = [
        sys.executable, '-u', str(SCRIPT),
        '--project-root', str(PROJECT_ROOT),
        '--n-jobs', '4',
        '--prepare-only',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Prepare-only execution skipped.')


## 4. Inspect preparation outputs

In [ ]:
RESULTS = PROJECT_ROOT / 'lda_results_v2'

flow = pd.read_csv(RESULTS/'tables'/'lda_v2_corpus_flow.csv')
print('=== CORPUS FLOW ===')
print(flow.to_string(index=False))

print('\n=== FAMILY CONTROL ===')
print((RESULTS/'provenance'/'near_duplicate_family_control.json').read_text(encoding='utf-8'))

fam_path = RESULTS/'tables'/'near_duplicate_family_summary.csv'
fam = pd.read_csv(fam_path)
print('\n=== LARGEST FAMILIES ===')
print(fam.sort_values('family_size', ascending=False).head(20).to_string(index=False) if len(fam) else 'No near-duplicate families found.')

terms = pd.read_csv(RESULTS/'tables'/'dictionary_term_filter_audit.csv')
print('\n=== DICTIONARY ACTION COUNTS ===')
print(terms['dictionary_action'].value_counts().to_string())

print('\n=== CORPUS / DICTIONARY STATS ===')
print((RESULTS/'provenance'/'corpus_and_dictionary_stats.json').read_text(encoding='utf-8'))


## 5. Full LDA search (60 runs)

For the shared server, the recommended method is the `nohup` command in `README_LDA_V2.md`. This cell is available for interactive execution but is **disabled by default** so the 60-run search cannot start accidentally.

In [ ]:
RUN_FULL_SEARCH = False

if RUN_FULL_SEARCH:
    cmd = [
        sys.executable, '-u', str(SCRIPT),
        '--project-root', str(PROJECT_ROOT),
        '--n-jobs', '4',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Full search NOT started. Set RUN_FULL_SEARCH=True only after reviewing the preparation outputs.')


## 6. Inspect final model-selection outputs

Run after `lda_results_v2/RUN_COMPLETE.json` exists.

In [ ]:
complete = RESULTS/'RUN_COMPLETE.json'
if not complete.exists():
    print('Full LDA v2 run is not complete yet.')
else:
    print(complete.read_text(encoding='utf-8'))
    print('\n=== K SEARCH SUMMARY ===')
    print(pd.read_csv(RESULTS/'tables'/'lda_k_search_summary.csv').to_string(index=False))
    print('\n=== SELECTION ===')
    print((RESULTS/'provenance'/'lda_selection.json').read_text(encoding='utf-8'))
    print('\n=== QUALITY GATE ===')
    print((RESULTS/'provenance'/'selected_solution_quality_gate.json').read_text(encoding='utf-8'))


## 7. Inspect selected topics

Topic names should be assigned only after reviewing **both** top terms and representative documents. Representative raw texts are internal review material.

In [ ]:
if complete.exists():
    terms = pd.read_csv(RESULTS/'tables'/'lda_topic_terms.csv')
    for tid, g in terms.groupby('topic_id'):
        print(f'\n=== Topic {tid} ===')
        print(', '.join(g.sort_values('rank')['term'].astype(str)))
    print('\n=== PREVALENCE ===')
    print(pd.read_csv(RESULTS/'tables'/'lda_topic_prevalence.csv').to_string(index=False))
else:
    print('Run the full search first.')
